In [1]:
# Importing required libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import json

In [2]:
# Load dataset
input_file_path = '../data/raw/train_data.csv'
data = pd.read_csv(input_file_path)
data['SalesDate'] = pd.to_datetime(data['SalesDate'])

In [3]:
# Select number of features.
number_of_features_required = 8


In [4]:
# Product specific metric for Product Mix Analysis
product_metrics = data.groupby('ProductID').agg(
    TotalSalesPrice=('ExtPrice', 'sum'),
    TotalCost=('ExtCost', 'sum'),
    TotalProfit=('Profit', 'sum'),
    AvgUnitPrice=('UnitPrice', 'mean'),
    AvgUnitCost=('UnitCost', 'mean'),
    TotalSalesQty=('SalesQty', 'sum'),
    TotalOrders=('OrderNumber', 'nunique'),
    UniqueCustomers=('CustomerName', 'nunique'),
    PriceVolatility=('UnitPrice', 'std')
).reset_index()

In [5]:
# Compute Additional Metrics
product_metrics['ProfitMargin'] = product_metrics['TotalProfit'] / product_metrics['TotalSalesPrice']
product_metrics['AvgOrderSize'] = product_metrics['TotalSalesQty'] / product_metrics['TotalOrders']
product_metrics['RepeatPurchaseRatio'] = product_metrics['TotalOrders'] / product_metrics['UniqueCustomers']

In [6]:
# Identify and Group customers based on product information
product_customers = data.groupby('ProductID')['CustomerName'].unique().reset_index()
product_customers['CustomerCount'] = product_customers['CustomerName'].apply(len)

In [7]:
# Merge Customer Data
product_metrics = product_metrics.merge(product_customers[['ProductID', 'CustomerCount']], on='ProductID', how='left')

In [8]:
# Calculate customer lifetime (duration between first and last purchase)
customer_lifetime = data.groupby("CustomerName").agg(
    FirstPurchase=("SalesDate", "min"),
    LastPurchase=("SalesDate", "max")
).reset_index()

# Compute lifetime in days
customer_lifetime["CustomerLifetimeDays"] = (customer_lifetime["LastPurchase"] - customer_lifetime["FirstPurchase"]).dt.days

# Merge with customer_metrics
customer_metrics = data.groupby("CustomerName").agg(
    TotalRevenue=("ExtPrice", "sum"), # total revenue generated by the customer
    TotalProfit=("Profit", "sum"), # total profit generated by the customer
    TotalUnitPrice=("UnitPrice", "sum"), # total individual unit price for all purchases by the customer
    TotalSalesQty=("SalesQty", "sum"), # total quantity sold to the customer
    OrderCount=("OrderNumber", "nunique"), # total number of orders placed by the customer
    NumDistinctProducts=("ProductID", "nunique"), # number of distinct products purchased by the customer
    NumDistinctProductGroups=("ProductGroupID", "nunique"), # number of distinct product groups purchased by the customer
    BranchCount=("BranchName", "nunique"), # number of branches visited by the customer
    TopProductID=("ProductID", lambda x: x.value_counts().idxmax()), # most popular product purchased by the customer
    TopProductGroupID=("ProductGroupID", lambda x: x.value_counts().idxmax()), # most popular product group purchased by the customer
    ProductDiversityIndex=("ProductID", lambda x: len(x.unique()) / x.count()), # number of distinct products divided by the total sales quantity
    ProductConcentrationIndex=("ExtPrice", lambda x: x.max() / x.sum()), # revenue of the most expensive product divided by the total revenue
    TopProductRevenueShare=("ExtPrice", lambda x: x.groupby(data["ProductID"]).sum().max() / x.sum()), # revenue share of the most expensive product
    TopProductGroupRevenueShare=("ExtPrice", lambda x: x.groupby(data["ProductGroupID"]).sum().max() / x.sum()) # revenue share of the most expensive product group
).reset_index()

customer_metrics = customer_metrics.merge(customer_lifetime[["CustomerName", "CustomerLifetimeDays"]], on="CustomerName", how="left")

In [9]:
# Compute additional features
customer_metrics['AvgOrderValue'] = customer_metrics['TotalRevenue'] / customer_metrics['OrderCount'] # average order value
customer_metrics['RepeatPurchaseRatio'] = customer_metrics['OrderCount'] / customer_metrics['NumDistinctProducts'] # raio of number of orders and number of distinct products bought

# Merge with Product-Level Profitability Metrics
product_profit = product_metrics[['ProductID', 'TotalProfit']]
data = data.merge(product_profit, on='ProductID', how='left')

# Identify High-Profit and Popular Products
high_profit_threshold = product_metrics['TotalProfit'].quantile(0.75)  # Top 25% most profitable
popular_product_threshold = product_metrics['TotalSalesQty'].quantile(0.75)  # Top 25% most sold

data['HighProfitProduct'] = data['Profit'] >= high_profit_threshold # Top 25% most profitable
data['PopularProduct'] = data['SalesQty'] >= popular_product_threshold # Top 25% most sold

# Compute Customer Engagement with High-Profit & Popular Products
customer_product_engagement = data.groupby('CustomerName').agg(
    HighProfitProductShare=('HighProfitProduct', 'mean'), # percentage of orders with high-profit products
    PopularProductShare=('PopularProduct', 'mean'), # percentage of orders with popular products
    HighProfitOrderRatio=('HighProfitProduct', 'sum'), # number of orders with high-profit products
    PopularOrderRatio=('PopularProduct', 'sum') # number of orders with popular products
).reset_index()

# Merge into customer dataset
customer_metrics = customer_metrics.merge(customer_product_engagement, on='CustomerName', how='left')

In [10]:
goal = 'TotalRevenue' 

# Select numeric features only
numeric_features = customer_metrics.select_dtypes(include=[np.number]).columns.tolist()

# Remove the target if included
if goal in numeric_features:
    numeric_features.remove(goal)

# Scale only the feature columns
scaler = StandardScaler()
customer_metrics[numeric_features] = scaler.fit_transform(customer_metrics[numeric_features])

# Train/test split
x = customer_metrics[numeric_features]
y = customer_metrics[goal]

In [11]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42) 

# Feature analysis
correlation = x.corrwith(y) # Pandas correalation
feature_correlation = pd.DataFrame(correlation, columns=["Correlation"]).sort_values('Correlation', ascending=False)
rf = RandomForestRegressor(n_estimators=100, random_state=42) # Train the Random Forest model
rf.fit(x_train, y_train)
accuracy = rf.score(x_test, y_test) # Accuracy of the Random Forest model before feature selection
feature_importance = pd.DataFrame(rf.feature_importances_, index=x_train.columns, columns=['importance']).sort_values('importance', ascending=False) # Feature importance

In [12]:

# Calculate final weights
feature_correlation["Correlation"] = np.abs(feature_correlation["Correlation"])
feature_correlation["Correlation"] /= feature_correlation["Correlation"].sum()
feature_importance["importance"] /= feature_importance["importance"].sum() # Normalize feature importance (ensure it sums to 1)
feature_weights = feature_importance.merge(feature_correlation, left_index=True, right_index=True) # Merge Importance and Correlation into a single DataFrame
alpha = 0.6  # Adjustable weight factor
feature_weights["FinalWeight"] = (alpha * feature_weights["importance"]) + ((1 - alpha) * feature_weights["Correlation"])
feature_weights["FinalWeight"] /= feature_weights["FinalWeight"].sum() # Normalize Final Weights
print("\nFeature Weights Based on Importance and Correlation:")
print(feature_weights.sort_values("FinalWeight", ascending=False))



Feature Weights Based on Importance and Correlation:
                             importance  Correlation  FinalWeight
TotalProfit                    0.393532     0.130633     0.288373
PopularOrderRatio              0.208643     0.113707     0.170669
TotalSalesQty                  0.172169     0.120596     0.151540
HighProfitOrderRatio           0.064318     0.119620     0.086439
OrderCount                     0.048883     0.115187     0.075405
TotalUnitPrice                 0.025719     0.089465     0.051217
NumDistinctProducts            0.018579     0.074417     0.040914
NumDistinctProductGroups       0.010309     0.059272     0.029894
ProductDiversityIndex          0.016842     0.027594     0.021143
PopularProductShare            0.014914     0.011818     0.013675
TopProductGroupRevenueShare    0.007813     0.021004     0.013090
TopProductRevenueShare         0.004543     0.024892     0.012682
ProductConcentrationIndex      0.007557     0.019497     0.012333
RepeatPurchaseRatio   

In [13]:
top_features = feature_weights.head(number_of_features_required).index.tolist()
print(f"Top {number_of_features_required} features based on Final Weights: {top_features}")

Top 8 features based on Final Weights: ['TotalProfit', 'PopularOrderRatio', 'TotalSalesQty', 'HighProfitOrderRatio', 'OrderCount', 'TotalUnitPrice', 'NumDistinctProducts', 'ProductDiversityIndex']


In [14]:
print(feature_weights)

                             importance  Correlation  FinalWeight
TotalProfit                    0.393532     0.130633     0.288373
PopularOrderRatio              0.208643     0.113707     0.170669
TotalSalesQty                  0.172169     0.120596     0.151540
HighProfitOrderRatio           0.064318     0.119620     0.086439
OrderCount                     0.048883     0.115187     0.075405
TotalUnitPrice                 0.025719     0.089465     0.051217
NumDistinctProducts            0.018579     0.074417     0.040914
ProductDiversityIndex          0.016842     0.027594     0.021143
PopularProductShare            0.014914     0.011818     0.013675
NumDistinctProductGroups       0.010309     0.059272     0.029894
TopProductGroupRevenueShare    0.007813     0.021004     0.013090
ProductConcentrationIndex      0.007557     0.019497     0.012333
TopProductRevenueShare         0.004543     0.024892     0.012682
RepeatPurchaseRatio            0.002381     0.023918     0.010996
AvgOrderVa

In [15]:
def save_feature_weights(feature_weights, output_file="feature_weights.csv", top_n=None, custom_features=None, normalize_subset=False):
    """
    Save feature weights to a CSV file.
    
    Parameters:
    - feature_weights (pd.DataFrame): DataFrame with feature weights (index: feature names, column: FinalWeight).
    - output_file (str): Name of the output CSV file.
    - top_n (int, optional): Number of top features to save (based on FinalWeight). If None, saves all.
    - custom_features (list, optional): List of specific feature names to save. Overrides top_n if provided.
    - normalize_subset (bool): If True, re-normalize the weights of the selected features to sum to 1.
    """
    # Select features to save
    if custom_features is not None:
        # Use custom feature list
        if not all(f in feature_weights.index for f in custom_features):
            raise ValueError("Some custom features not found in feature_weights index")
        df_to_save = feature_weights.loc[custom_features, ["FinalWeight"]]
    elif top_n is not None:
        # Use top N features
        df_to_save = feature_weights[["FinalWeight"]].nlargest(top_n, "FinalWeight")
    else:
        # Save all features
        df_to_save = feature_weights[["FinalWeight"]]
    
    # Re-normalize weights if requested
    if normalize_subset and not df_to_save.empty:
        df_to_save["FinalWeight"] = df_to_save["FinalWeight"] / df_to_save["FinalWeight"].sum()
    
    # Save to CSV
    df_to_save.to_csv(output_file, index=True, index_label="Feature")
    
    # Print confirmation
    print(f"Saved features and weights to '{output_file}'")
    print(f"Number of features saved: {len(df_to_save)}")
    print(f"Sum of weights: {df_to_save['FinalWeight'].sum():.6f}")


In [16]:
save_feature_weights(feature_weights, output_file="../data/features_with_weights.csv", top_n=8, normalize_subset=True)

Saved features and weights to '../data/features_with_weights.csv'
Number of features saved: 8
Sum of weights: 1.000000
